In [1]:
import pandas as pd
from nltk.corpus import stopwords
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
import numpy as np

In [2]:
stop_words = set(stopwords.words('english'))

In [3]:
#Nettoyage des données
def text_process(mess):
    lower_mess = mess.lower()
    nopunc = [char for char in lower_mess if char not in string.punctuation]
    nopunc = ''.join(nopunc)
    clean_mess = [word for word in nopunc.split() if word not in stop_words]
    return clean_mess

In [47]:
data = pd.read_csv('../../data/train_with_sentiments.csv')
data = data.dropna()
data = data.drop_duplicates(subset=['Context','Response'])
data = data.drop_duplicates(subset=['Context'])
data = data.drop_duplicates(subset=['Response'])
test = data.sample(n=10, random_state=42)
data = data.drop(test.index)
data.to_csv('../../data/train_unique_e2p2.csv', index=False)
test.to_csv('../../data/test.csv', index=False)

In [50]:
train_data = pd.read_csv('../../data/train_unique_e2p2.csv')
test_data = pd.read_csv('../../data/test.csv')

In [51]:
train_data["Context_clean"] = train_data["Context"].apply(text_process)
train_data["Response_clean"] = train_data["Response"].apply(text_process)
# Structure du dictionnaire : mot -> liste des discussions contenant ce type de mot
index_inverse = {}
# Nombre minimum d'occurrences pour garder une discussion
SEUIL = 1
for idx, row in train_data.iterrows():
    question = row["Context"]
    response = row["Response"]
    #combine les mots de la question et de la réponse
    mots = row["Context_clean"] + row["Response_clean"]
    # Compter le nombre d'apparitions de chaque mot
    from collections import Counter
    compteur = Counter(mots)
    for mot,count in compteur.items():
        if count >= SEUIL:
        #Si le mot n'existe pas encore dans le dictionnaire, on l'initialise
            if mot not in index_inverse:
                index_inverse[mot] = []
            #On ajoute la discussion associée à ce mot dans la liste
            index_inverse[mot].append({
                "id": idx,
                "question": question,
                "response": response
            })

In [52]:
anxiete = index_inverse.get("anxiety", [])
print("Nombre de discussion contenant le mot anxiety plus de",SEUIL,"fois :", len(anxiete))
print("\nDiscussion pour le mot 'anxiety' :\n")
#Pour un affichage plus propre
for i, item in enumerate(anxiete, 1):
    print(f"--- Exemple {i} ---")
    print(f"ID       : {item['id']}")
    print(f"Question : {item['question']}")
    print(f"Réponse  : {item['response']}\n")

Nombre de discussion contenant le mot anxiety plus de 1 fois : 116

Discussion pour le mot 'anxiety' :

--- Exemple 1 ---
ID       : 1
Question : I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac.    I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years.
   I’ve never had counseling about any of this. Do I have too many issues to address in counseling?
Réponse  : Let me start by saying there are never too many concerns that you can bring into counselling. In fact, most people who come to see me for counselling have more than one issue they would like to work on in psychotherapy and most times these are all interconnected. In counselling, we work together, collaboratively, to figure out which issues you would like to address first and then together we develop an individualized plan of care. Basically, it’s like a road map 

In [55]:
# Méthode 1 : TF-IDF
vectorizer = TfidfVectorizer(analyzer=text_process)
#X_train = vectorizer.fit_transform(train_data['Context'])
def questionQuestion(question,k=5):
    sentiment = test_data.loc[test_data['Context'] == question, 'predicted_sentiment_tfidf'].values[0]
    data_sentiment = train_data[train_data['predicted_sentiment_tfidf'] == sentiment]
    X_train = vectorizer.fit_transform(data_sentiment['Context'])
    X_question = vectorizer.transform([question])
    similarities = cosine_similarity(X_question, X_train).flatten()
    top_indices = similarities.argsort()[::-1][:k]
    list = []
    for idx in top_indices:
        list.append((train_data.iloc[idx]["Response"], similarities[idx]))
    return list

In [56]:
# Test méthode 1 : TF-IDF
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    list = questionQuestion(question)
    for predicted_response, score in list:
        print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: He was in love with someone years ago, and he still thinks about her time to time. He said, and I quote, "That relationship is definitely over. I love you, but that girl will always be in my mind." It just didn't feel like he appreciated all the things I've done to make him happy.
Actual Response: Trust your intuition on your conclusion about this guy.He may very well love you, only with the ex so prominent in his mind, it is possible your feeling of not being appreciated now, would multiply if ever the two of you needed to address a delicate topic.Since he is emotionally attached to the former gf, it is very likely he wouldn't be able to fully love you as much as you'd like and are already sensing.
Top response:
- (Cosine Similarity: 0.3655) This can be a difficult situation.  Typically, only animals that are specifically trains to accomplish a specific task are legally protected as Service Animsls. Even though that can be very helpful, emotional support animals are not gene

- (Cosine Similarity: 0.3956) Hi Winters, I'm so glad you wrote, because I think there are a lot of young women experiencing the exact same thing. You feel self-loathing for both being a virgin, and for being sexually active. Young women have always gotten crazy mixed messages about what they're supposed to be. They feel pressure to be pure, and they also feel pressure to be the vixen and please men sexually. But you can't be both, so you can't ever win if you buy into all that horse manure (excuse my language). This current hook-up culture puts added pressure on girls to expect nothing more than random sexual encounters that leave you feeling empty and used; perhaps desirable in that moment but mostly worthless. The stupid part is that research tells us that young men are also impacted negatively by this cultural norm that values sex and not relationship; they feel guilt, and loneliness.  I urge you to talk to other girls and women about your feelings. My hope and prayer is that they 

In [58]:
# Méthode 1 : Word2Vec
phrases_train = train_data["Context"].apply(text_process).tolist()
model_w2v = Word2Vec(sentences=phrases_train, vector_size=100, window=5, min_count=1, workers=4)
def vectoriser_phrase(phrase):
    mots = text_process(phrase)
    vecteurs = [model_w2v.wv[mot] for mot in mots if mot in model_w2v.wv]
    if vecteurs:
        return np.mean(vecteurs, axis=0)
    else:
        return np.zeros(model_w2v.vector_size)
# X_train_w2v = np.array([vectoriser_phrase(phrase) for phrase in train_data["Context"]])
def questionQuestion_w2v(question,k=5):
    sentiment = test_data.loc[test_data['Context'] == question, 'predicted_sentiment_tfidf'].values[0]
    data_sentiment = train_data[train_data['predicted_sentiment_tfidf'] == sentiment]
    X_train_w2v = np.array([vectoriser_phrase(phrase) for phrase in data_sentiment["Context"]])
    vecteur_question = vectoriser_phrase(question).reshape(1, -1)
    similarities = cosine_similarity(vecteur_question, X_train_w2v).flatten()
    top_indices = similarities.argsort()[::-1][:k]
    list = []
    for idx in top_indices:
        list.append((train_data.iloc[idx]["Response"], similarities[idx]))
    return list


In [59]:
# Test méthode 1 : Word2Vec
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    list = questionQuestion_w2v(question)
    for predicted_response, score in list:
        print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: He was in love with someone years ago, and he still thinks about her time to time. He said, and I quote, "That relationship is definitely over. I love you, but that girl will always be in my mind." It just didn't feel like he appreciated all the things I've done to make him happy.
Actual Response: Trust your intuition on your conclusion about this guy.He may very well love you, only with the ex so prominent in his mind, it is possible your feeling of not being appreciated now, would multiply if ever the two of you needed to address a delicate topic.Since he is emotionally attached to the former gf, it is very likely he wouldn't be able to fully love you as much as you'd like and are already sensing.
Top response:
- (Cosine Similarity: 1.0000) Hello, and thank you for your question. First, I want to tell you how sorry I am for the experience you had with your parents. That is a grief and trauma that is certainly hard to imagine. Trauma and grief can affect us in many ways, and

- (Cosine Similarity: 1.0000) Maybe you need more time to reflect and organize your thoughts.Try to figure out what would make you feel more relaxed about talking to your dad or stepmom.Also its possible you simply don't feel safe around either of them and so intuitively realize you're better off not talking with them about a delicate matter.Depending on whether you trust talking to dad and stepmom, you may simply wish to excuse yourself from speaking about yourself.There's no good reason to be heartfelt with people whom you don't feel are willing to accept or understand who you are.
- (Cosine Similarity: 1.0000) Answers about our inner lives are most successfully reached from a sense of feeling grounded in oneself.First step is to accept your nervousness and restless sleep.  As often as possible, sleep during daytimes in order for your body to catch up on its need for rest.Accept too about feeling down.  It is normal to feel down once in a while.  From this place of self-acceptance, t

In [61]:
# Méthode 1 : Bert
model = SentenceTransformer('all-MiniLM-L6-v2')
#question_bert = model.encode(train_data["Context"].tolist())
def questionQuestion_bert(question,k=5):
    sentiment = test_data.loc[test_data['Context'] == question, 'predicted_sentiment_tfidf'].values[0]
    data_sentiment = train_data[train_data['predicted_sentiment_tfidf'] == sentiment]
    question_bert = model.encode(data_sentiment["Context"].tolist())
    question_embedding = model.encode([question])
    cosine_similarities = cosine_similarity(question_embedding,question_bert).flatten()
    top_k_indices = cosine_similarities.argsort()[::-1][:k]
    list = []
    for idx in top_k_indices:
        list.append((train_data.iloc[idx]["Response"], cosine_similarities[idx]))
    return list

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [62]:
# Test méthode 1 : Bert
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    list = questionQuestion_bert(question)
    for predicted_response, score in list:
        print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: He was in love with someone years ago, and he still thinks about her time to time. He said, and I quote, "That relationship is definitely over. I love you, but that girl will always be in my mind." It just didn't feel like he appreciated all the things I've done to make him happy.
Actual Response: Trust your intuition on your conclusion about this guy.He may very well love you, only with the ex so prominent in his mind, it is possible your feeling of not being appreciated now, would multiply if ever the two of you needed to address a delicate topic.Since he is emotionally attached to the former gf, it is very likely he wouldn't be able to fully love you as much as you'd like and are already sensing.
Top response:


- (Cosine Similarity: 0.4646) How do you help yourself to believe you require more than what he offers to you?What do you get from this relationship which feels satisfying?To answer this question may in the longterm be the best way to help your bf.
- (Cosine Similarity: 0.4565) The fact that you're reaching out says that there is something in you that wants this to be different, and that drive might be something worth tapping into. "Why do I keep trying?" is a question that might give you some insight into what it is in you that keeps you going. A lot of therapists/counselors are now offering video therapy. As long as you're in the same state as a therapist offering this service, you could connect with someone helpful from the comfort of your home, even being in your small town. I'd recommend looking into this option, because you're asking a lot of really deep questions and might benefit from having those conversations with someone who can help you find your own answers.
- (Cosine Simi

In [64]:
# Méthde 2 : TF-IDF 
vectorizer = TfidfVectorizer(analyzer=text_process)
# X_train = vectorizer.fit_transform(train_data['Response'])
def questionQuestion2(question,k=5):
    sentiment = test_data.loc[test_data['Context'] == question, 'predicted_sentiment_tfidf'].values[0]
    data_sentiment = train_data[train_data['predicted_sentiment_tfidf'] == sentiment]
    X_train = vectorizer.fit_transform(data_sentiment['Response'])
    question_vector = vectorizer.transform([question])
    similarities = cosine_similarity(question_vector, X_train).flatten()
    top_indices = similarities.argsort()[::-1][:k]
    list = []
    for idx in top_indices:
        list.append((train_data.iloc[idx]["Response"], similarities[idx]))
    return list

In [65]:
# Test méthode 2 : TF-IDF
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    list = questionQuestion2(question)
    for predicted_response, score in list:
        print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: He was in love with someone years ago, and he still thinks about her time to time. He said, and I quote, "That relationship is definitely over. I love you, but that girl will always be in my mind." It just didn't feel like he appreciated all the things I've done to make him happy.
Actual Response: Trust your intuition on your conclusion about this guy.He may very well love you, only with the ex so prominent in his mind, it is possible your feeling of not being appreciated now, would multiply if ever the two of you needed to address a delicate topic.Since he is emotionally attached to the former gf, it is very likely he wouldn't be able to fully love you as much as you'd like and are already sensing.
Top response:
- (Cosine Similarity: 0.1407) Being tired can really affect almost everyone's ability to work through things that make them sad, confused, or angry, among other emotions. If you're having difficulty sleeping, try to get into a habit of going to bed and waking up clos

- (Cosine Similarity: 0.3541) You've already taken the first step. You want to not hate yourself. Self-acceptance is hard! And it's on a spectrum. On one side we have self-hate, on the other extreme; self-love. And then, there is all this stuff in the middle. It kind of looks like thisAnd working toward self-love often means moving around through all these. Becoming aware of your emotions, exploring the parts of you you that easier and harder to accept, self-kindness, self-forgiveness, self-compassion and ultimately self-love. It is a recovery process and has to be an active thing each day. Meeting with a counselor can give you a partner in that process. Your counselor can also help you to recognize pieces that may be more difficult to see from your eyes as they have an outside view. And with self-acceptance, confidence comes naturally although you may need to practice behaviors that show assertiveness, confidence and boundaries that protect you. Wishing you the absolute best with this

In [66]:
# Méthode 2 : Word2Vec
phrases_train = train_data["Response"].apply(text_process).tolist()
model_w2v = Word2Vec(sentences=phrases_train, vector_size=100, window=5, min_count=2)
def vectoriser_phrase2(phrase):
    mots = text_process(phrase)
    vecteurs = [model_w2v.wv[mot] for mot in mots if mot in model_w2v.wv]
    if vecteurs:
        return np.mean(vecteurs, axis=0)
    else:
        return np.zeros(model_w2v.vector_size)
# X_train_w2v = np.array([vectoriser_phrase2(phrase) for phrase in train_data["Response"]])
def questionQuestion2_w2v(question,k=5):
    sentiment = test_data.loc[test_data['Context'] == question, 'predicted_sentiment_tfidf'].values[0]
    data_sentiment = train_data[train_data['predicted_sentiment_tfidf'] == sentiment]
    X_train_w2v = np.array([vectoriser_phrase2(phrase) for phrase in data_sentiment["Response"]])
    vecteur_question = vectoriser_phrase2(question).reshape(1, -1)
    similarities = cosine_similarity(vecteur_question, X_train_w2v).flatten()
    top_indices = similarities.argsort()[::-1][:k]
    results = []
    for idx in top_indices:
        results.append((train_data.iloc[idx]["Response"], similarities[idx]))
    return results


In [67]:
# Test méthode 2 : Word2Vec
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    list = questionQuestion2_w2v(question)
    for predicted_response, score in list:
        print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: He was in love with someone years ago, and he still thinks about her time to time. He said, and I quote, "That relationship is definitely over. I love you, but that girl will always be in my mind." It just didn't feel like he appreciated all the things I've done to make him happy.
Actual Response: Trust your intuition on your conclusion about this guy.He may very well love you, only with the ex so prominent in his mind, it is possible your feeling of not being appreciated now, would multiply if ever the two of you needed to address a delicate topic.Since he is emotionally attached to the former gf, it is very likely he wouldn't be able to fully love you as much as you'd like and are already sensing.
Top response:
- (Cosine Similarity: 1.0000) Hello,While one can be sad from time to time, feeling sad "all the time" could be a sign of depression. If you feel sad on most days, it is worthwhile speaking to a psychologist to determine whether you suffer clinical depression. Feelin

- (Cosine Similarity: 1.0000) Maybe you have depression.The name of your condition matters much less than the descriptions you wrote of how you feel.Since you've observed how you sometimes interact with people and realize you aren't happy with the result, you've a very solid starting point for reflecting on your deeper wishes in relating to others.Start with asking reasons of yourself about the puzzling aspects of how you're engaging with others.Theorizing as to "why" you feel that pushing people away is "easier", and easier than what?Googling the keywords of how you feel, may open a starting point for ideas on knowing yourself and what you wish for.
- (Cosine Similarity: 1.0000) The problem you describe sounds very wearing on your spirit.Are there particular reasons for why you feel everyone hates you?Have you been in a clash of ideas or opinions and feel yourself in the minority viewpoint?Or does your sense of being shut out start within your own mind, as though you anticipate that o

In [68]:
# Méthode 2 : Bert
model = SentenceTransformer('all-MiniLM-L6-v2')
# question_bert = model.encode(train_data["Response"].tolist())
def questionQuestion2_bert(question,k=5):
    sentiment = test_data.loc[test_data['Context'] == question, 'predicted_sentiment_tfidf'].values[0]
    data_sentiment = train_data[train_data['predicted_sentiment_tfidf'] == sentiment]
    question_bert = model.encode(data_sentiment["Response"].tolist())
    question_embedding = model.encode([question])
    cosine_similarities = cosine_similarity(question_embedding,question_bert).flatten()
    top_k_indices = cosine_similarities.argsort()[::-1][:k]
    res = []
    for idx in top_k_indices:
        res.append((train_data.iloc[idx]["Response"], cosine_similarities[idx]))
    return res

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [69]:
# Test méthode 2 : Bert
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    list = questionQuestion2_bert(question)
    for predicted_response, score in list:
        print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: He was in love with someone years ago, and he still thinks about her time to time. He said, and I quote, "That relationship is definitely over. I love you, but that girl will always be in my mind." It just didn't feel like he appreciated all the things I've done to make him happy.
Actual Response: Trust your intuition on your conclusion about this guy.He may very well love you, only with the ex so prominent in his mind, it is possible your feeling of not being appreciated now, would multiply if ever the two of you needed to address a delicate topic.Since he is emotionally attached to the former gf, it is very likely he wouldn't be able to fully love you as much as you'd like and are already sensing.
Top response:


- (Cosine Similarity: 0.4836) If you are currently feeling as if you want to die, please call 800-273-8255 and talk to someone.One way to work on not always thinking so negatively about yourself is to surround yourself with people who are more positive toward you. Do you have friends or family who are supportive?Can you find one part of you that you do not think is ugly?If your stretch marks are still bothering you, talk with a pharmacist or your primary care physician. Sometimes there are creams or lotions you can use to decrease stretch marks and they should be able to guide you in the right direction.You mentioned mostly physical things here. I wonder if you can find one small thing each day that is going right and build from there. Perhaps your son makes you smile?
- (Cosine Similarity: 0.3931) The fact that you're reaching out says that there is something in you that wants this to be different, and that drive might be something worth tapping into. "Why do I keep trying?" is a ques

In [75]:
X = model.encode(test_data['Response'].tolist())
def eval_mrr_bert(predictions):
    mrr_total = 0
    for i in range(len(test_data)):
        actualResponse = X[i].reshape(1, -1)
        predictedResponse = model.encode([predictions[i]])[0].reshape(1, -1)
        similarity = cosine_similarity(predictedResponse, actualResponse)[0]
        score = similarity.max()
        print(score)
        if score > 0:
            mrr_total += score
    return mrr_total / len(test_data)
predictions = []
for question in test_data['Context']:
    predicted_response = questionQuestion2_bert(question)[0][0]
    predictions.append(predicted_response)
res = eval_mrr_bert(predictions)
print(f"MRR: {res:.4f}")

0.082087085
0.12281275
0.28661555
0.1163989
0.38678294
0.28954342
0.56203806
0.4126962
0.35827258
0.29754907
MRR: 0.2915
